<a href="https://colab.research.google.com/github/IBM/vLLM-Hook/blob/main/notebooks/demo_actsteer_no_comma_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# No-Comma Activation Steering

vLLM-Hook is an extensible framework that aims to allow selective access to model internals during inference.
In this notebook, we demonstrate a Phi-3 activation-steering vector that nudges generations away from comma usage.

The steering vector was derived from real Phi-3 hidden states using no-comma instruction-following examples. The notebook compares the same prompts with and without the hook enabled and counts commas in each output.


### Installation

In Colab, run this setup cell first from a fresh GPU runtime. It clones the repository, installs the requirements, installs `vllm_hook_plugins` in editable mode, and switches into the repository root so relative steering-vector paths resolve correctly.


In [ ]:
from pathlib import Path
import importlib
import importlib.util
import os
import shutil
import subprocess
import sys

REPO_URL = "https://github.com/IBM/vLLM-Hook.git"
REPO_BRANCH = "main"
REPO_NAME = "vLLM-Hook"

IN_COLAB = "google.colab" in sys.modules
NOTEBOOK_DIR = Path.cwd()


def _repo_remote_matches(repo_root: Path, expected_remote: str) -> bool:
    try:
        origin_url = subprocess.run(
            ["git", "-C", str(repo_root), "remote", "get-url", "origin"],
            check=True,
            capture_output=True,
            text=True,
        ).stdout.strip().removesuffix(".git")
    except Exception:
        return False
    return origin_url == expected_remote


def _find_existing_repo_root(start_dir: Path, expected_remote: str):
    for candidate in [start_dir, *start_dir.parents]:
        if (candidate / ".git").exists() and _repo_remote_matches(candidate, expected_remote):
            return candidate
    return None


if IN_COLAB:
    expected_remote = REPO_URL.removesuffix(".git")
    existing_repo_root = _find_existing_repo_root(NOTEBOOK_DIR, expected_remote)
    if existing_repo_root is not None:
        REPO_ROOT = existing_repo_root
        print(f"Colab detected. Reusing existing repo at {REPO_ROOT}")
    else:
        REPO_ROOT = Path("/content") / REPO_NAME
        if not REPO_ROOT.exists():
            print(f"Colab detected. Cloning {REPO_URL} ({REPO_BRANCH}) into {REPO_ROOT} ...")
            subprocess.run(["git", "clone", "--branch", REPO_BRANCH, REPO_URL, str(REPO_ROOT)], check=True)
        elif not _repo_remote_matches(REPO_ROOT, expected_remote):
            print(f"Remote mismatch under {REPO_ROOT}; replacing clone with {expected_remote}")
            shutil.rmtree(REPO_ROOT)
            subprocess.run(["git", "clone", "--branch", REPO_BRANCH, REPO_URL, str(REPO_ROOT)], check=True)
        else:
            print(f"Colab detected. Reusing existing clone at {REPO_ROOT}")
            subprocess.run(["git", "-C", str(REPO_ROOT), "fetch", "origin", REPO_BRANCH], check=True)
            subprocess.run(["git", "-C", str(REPO_ROOT), "checkout", REPO_BRANCH], check=True)
            subprocess.run(["git", "-C", str(REPO_ROOT), "pull", "--ff-only", "origin", REPO_BRANCH], check=True)
else:
    REPO_ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == "notebooks" else NOTEBOOK_DIR

os.chdir(REPO_ROOT)

PKG_DIR = REPO_ROOT / "vllm_hook_plugins"
REQ_FILE = REPO_ROOT / "requirement.txt"
RUN_INSTALL = IN_COLAB

print("Colab      :", IN_COLAB)
print("Repo root   :", REPO_ROOT)
print("Repo branch :", REPO_BRANCH)
print("Package dir :", PKG_DIR)
print("Req file    :", REQ_FILE)

if IN_COLAB:
    try:
        import torch
    except Exception:
        torch = None
    has_cuda = bool(torch is not None and torch.cuda.is_available())
    has_cudart = importlib.util.find_spec("nvidia.cuda_runtime") is not None
    if not has_cuda and not has_cudart:
        raise RuntimeError(
            "This notebook requires a Colab GPU runtime with CUDA available. "
            "Use Runtime > Change runtime type > GPU, then rerun from a fresh runtime."
        )

if RUN_INSTALL:
    if REQ_FILE.exists():
        req_cmd = [sys.executable, "-m", "pip", "install", "-r", str(REQ_FILE)]
        print("Running:", " ".join(req_cmd))
        subprocess.run(req_cmd, check=True)
    else:
        print("Warning: requirement.txt not found; skipping dependency install.")

    protobuf_cmd = [sys.executable, "-m", "pip", "install", "--force-reinstall", "protobuf>=5.29.6,<6.30"]
    print("Running:", " ".join(protobuf_cmd))
    subprocess.run(protobuf_cmd, check=True)

    editable_cmd = [sys.executable, "-m", "pip", "install", "-e", str(PKG_DIR)]
    print("Running:", " ".join(editable_cmd))
    subprocess.run(editable_cmd, check=True)
else:
    print("Install step skipped outside Colab. Set RUN_INSTALL=True if dependencies are not installed.")

plugin_src_dir = str(PKG_DIR.resolve())
if plugin_src_dir not in sys.path:
    sys.path.insert(0, plugin_src_dir)
importlib.invalidate_caches()
print("Plugin source:", plugin_src_dir)
print("Python exec  :", sys.executable)


### Importing the Hook-Enabled LLM

The plugin provides its own LLM wrapper that behaves like `vllm.LLM` but adds support for hooks and instrumentation.


In [ ]:
from vllm_hook_plugins import HookLLM


### Environment & Multiprocessing Setup

In [ ]:
import gc
import io
import os
import multiprocessing as mp
import sys
from pathlib import Path

import torch
from vllm import SamplingParams

IN_COLAB = "google.colab" in sys.modules
os.environ["VLLM_USE_V1"] = "1"

if IN_COLAB:
    mp.set_start_method("fork", force=True)
    os.environ["VLLM_WORKER_MULTIPROC_METHOD"] = "fork"
    os.environ["VLLM_ENABLE_V1_MULTIPROCESSING"] = "0"
    os.environ.setdefault("HF_HOME", "/content/.cache/huggingface")
    os.environ.setdefault("HUGGINGFACE_HUB_CACHE", "/content/.cache/huggingface/hub")
    os.makedirs(os.environ["HUGGINGFACE_HUB_CACHE"], exist_ok=True)
    os.makedirs("/content/.cache/vllm-hook", exist_ok=True)

    def _patch_fileno(stream, fallback_stream, fallback_fd):
        try:
            stream.fileno()
        except io.UnsupportedOperation:
            def _fileno():
                try:
                    return fallback_stream.fileno()
                except Exception:
                    return fallback_fd
            stream.fileno = _fileno

    _patch_fileno(sys.stdout, sys.__stdout__, 1)
    _patch_fileno(sys.stderr, sys.__stderr__, 2)
    print("Colab detected: using fork start method and disabled V1 multiprocessing")
else:
    mp.set_start_method("spawn", force=True)
    os.environ["VLLM_WORKER_MULTIPROC_METHOD"] = "spawn"

try:
    REPO_ROOT
except NameError:
    REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

os.chdir(REPO_ROOT)
print("Working directory:", Path.cwd())
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU count:", torch.cuda.device_count())
    print("GPU name :", torch.cuda.get_device_name(0))


### Initialize `HookLLM`

The no-comma steering config uses `add_vector` on Phi-3 layer 15 with a coefficient of 40 and applies the vector to all tokens.


In [ ]:
import json

cache_dir = "/content/.cache/vllm-hook" if "google.colab" in sys.modules else "./cache/"
model = "microsoft/Phi-3-mini-4k-instruct"
json_path = Path("model_configs/activation_steer/Phi-3-mini-4k-instruct-no-comma.json")

with open(json_path, "r") as f:
    config = json.load(f)

vector_path = Path(config["steering"]["vector_path"])
print(json.dumps(config, indent=2))
print("Config file    :", json_path)
print("Steering vector:", vector_path, "exists=", vector_path.exists())


Before rebuilding `HookLLM`, clear any stale CUDA allocations left by earlier notebook runs in the same Colab session.

In [ ]:
try:
    del llm
except NameError:
    pass

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()


In [ ]:
llm_kwargs = dict(
    model=model,
    worker_name="steer_hook_act",
    config_file=str(json_path),
    download_dir=cache_dir,
    trust_remote_code=True,
    dtype="auto",
    enforce_eager=True,
    enable_hook=True,
    tensor_parallel_size=1,
)

if IN_COLAB:
    llm_kwargs.update(
        gpu_memory_utilization=0.5,
        max_model_len=1024,
        max_num_seqs=1,
        enable_prefix_caching=False,
        compilation_config={"mode": 0},
    )
else:
    llm_kwargs.update(
        gpu_memory_utilization=0.6,
        max_model_len=2048,
        enable_prefix_caching=True,
    )

llm = HookLLM(**llm_kwargs)


### Test Cases

Each prompt asks for a useful response without commas. The comparison below runs the same prompts with and without the activation-steering hook.


In [ ]:
def comma_count(text: str) -> int:
    return text.count(",")


prompts = [
    (
        "I am planning a trip to Japan and I would like thee to write an itinerary "
        "for my journey in a Shakespearean style. You are not allowed to use any "
        "commas in your response."
    ),
    (
        "Write a short product profile for a travel mug that keeps coffee hot for "
        "a full workday. Do not use any commas in your response."
    ),
]

prompts


### Sampling Parameters

Token `32007` is Phi-specific and is included as an additional stop token, matching the existing Phi-3 activation-steering demo.


In [ ]:
sampling_params = SamplingParams(
    temperature=0.0,
    max_tokens=160 if IN_COLAB else 220,
    stop_token_ids=[llm.tokenizer.eos_token_id, 32007],
)


### Generate With and Without Steering

In [ ]:
examples = [
    llm.tokenizer.apply_chat_template(
        [{"role": "user", "content": prompt}],
        add_generation_prompt=True,
        tokenize=False,
    )
    for prompt in prompts
]

steered_outputs = llm.generate(examples, sampling_params)
llm.llm_engine.reset_prefix_cache()
baseline_outputs = llm.generate(examples, sampling_params, use_hook=False)
llm.llm_engine.reset_prefix_cache()


### Results

In [ ]:
rows = []

for prompt, steered, baseline in zip(prompts, steered_outputs, baseline_outputs):
    steered_text = steered.outputs[0].text
    baseline_text = baseline.outputs[0].text
    rows.append(
        {
            "prompt": prompt,
            "steered_commas": comma_count(steered_text),
            "baseline_commas": comma_count(baseline_text),
        }
    )

    print("=" * 80)
    print("[Prompt]")
    print(prompt)

    print("\n[With no-comma activation steering]")
    print(steered_text)
    print(f"Commas: {comma_count(steered_text)}")

    print("\n[Without activation steering]")
    print(baseline_text)
    print(f"Commas: {comma_count(baseline_text)}")

print("=" * 80)
print("Summary:")
for row in rows:
    print(
        f"steered_commas={row['steered_commas']} | "
        f"baseline_commas={row['baseline_commas']} | "
        f"prompt={row['prompt'][:70]}..."
    )
